# KDE 模拟配置：国泰君安递推式的检验与修正

**来源**：国泰君安数量化专题之一百零九《资产配置之步步为营：尾部风险控制与优化》（2018-03-12）。
**未能取得原文全文**（仅在需登录的资料站），方法取自整理稿 4.2.2 节：

1. 对各资产历史收益做高斯核密度估计（KDE），独立抽样得到 K；
2. 对历史协方差做 Cholesky 分解 Σ = AAᵀ，用递推式注入相关：
   X_i = Σ_{j<i} (K_j − μ_j)/A_jj · A_ij + K_i
3. 以过去 120 个交易日为输入，每次模拟 1500 次，寻找风险度量最小的组合权重。

整理稿称该递推式「保证模拟协方差与历史一致、边缘分布与原资产一致」。
整理稿 4.2.2 节已从公式上指出两点都不成立；本 notebook 在真实数据上量化它的影响，并与两种正确做法对比：

| 方法 | 做法 | 边缘分布 | 相关结构 |
|---|---|---|---|
| 国泰君安递推 | 见上式 | 被卷积改变，**依赖资产排列顺序** | 方差被高估 |
| 高斯 Copula + KDE | 正态得分相关 → 多元正态 → Φ → 各资产 KDE 分位数逆变换 | 保留 | 目标秩相关 |
| Iman-Conover | 各列独立从 KDE 抽样，只按目标秩相关重排顺序 | 保留（抽样值不变） | 目标秩相关 |

**风险度量**：原文的「期内风险 + 期末风险（基于回撤和达标概率）」在整理稿中没有定义，
本研究改用**模拟的 20 日期末收益的 95% ES**，用线性规划求最小值。因此不复现原文的收益数字。

## 0 · 环境与数据

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tailrisk").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kurtosis, ks_2samp, skew
from tailrisk import data, simulation as sm, allocation as al, backtest as bt
from tailrisk.marginals import es_hist

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({
    "figure.figsize": (13, 4),
    "axes.unicode_minus": False,
    "font.sans-serif": ["PingFang SC", "Heiti TC", "Arial Unicode MS", "SimHei", "DejaVu Sans"],
})
WINDOW, HORIZON, N_PATHS = 120, 20, 1500
rng = np.random.default_rng(2018)

原文五资产为沪深300、恒生指数、标普500、中债国债总财富指数、上海金交所 Au99.99。
恒生、标普 500 与 A 股收盘时间不同，免费的 Au99.99 只有 2016 年以后，这里一律换成在 A 股交易时段收盘的 ETF，
债券用中债综合财富指数（与 A 股同日收盘）。共同区间因此从 2014 年开始。

In [2]:
ASSETS5 = ["沪深300ETF", "恒生ETF", "标普500ETF", "中债综合财富", "黄金ETF"]
R5 = data.to_returns(data.load_prices(ASSETS5)).dropna()
R2 = data.to_returns(data.load_prices(["沪深300", "中债综合财富"]).loc["2005-06-01":]).dropna()
print(f"五资产 {R5.index.min():%Y-%m-%d} ~ {R5.index.max():%Y-%m-%d}，{len(R5)} 天；股债两资产自 {R2.index.min():%Y-%m-%d}")
R5.corr().style.format("{:.2f}")

五资产 2014-01-16 ~ 2026-10-12，3092 天；股债两资产自 2005-06-02


,沪深300ETF,恒生ETF,标普500ETF,中债综合财富,黄金ETF
沪深300ETF,1.00,0.60,0.26,-0.12,0.06
恒生ETF,0.60,1.00,0.41,-0.12,0.05
标普500ETF,0.26,0.41,1.00,-0.04,0.04
中债综合财富,-0.12,-0.12,-0.04,1.00,0.08
黄金ETF,0.06,0.05,0.04,0.08,1.00


---
## 1 · 在已知答案的数据上检查

人工构造三个偏态厚尾、两两相关的资产（对数正态变换），各方法模拟 10 万次，
与真实分布比较协方差、偏度、峰度和 KS 距离（KS 越小，边缘分布越接近）。

In [3]:
T = 2000
z = rng.multivariate_normal([0, 0, 0], [[1, .6, .3], [.6, 1, .4], [.3, .4, 1]], T)
H = np.exp(0.6 * z)
H = (H - H.mean(0)) / H.std(0) * np.array([.01, .02, .015])
C0 = np.cov(H, rowvar=False)
rows = []
for name, f in sm.METHODS.items():
    X = f(H, 100_000, rng)
    C = np.cov(X, rowvar=False)
    for j in range(3):
        rows.append({"方法": name, "资产": f"第{j + 1}位", "方差误差": C[j, j] / C0[j, j] - 1,
                     "偏度（真实）": skew(H[:, j]), "偏度（模拟）": skew(X[:, j]),
                     "峰度（真实）": kurtosis(H[:, j]), "峰度（模拟）": kurtosis(X[:, j]),
                     "KS 距离": ks_2samp(X[:, j], H[:, j]).statistic})
pd.DataFrame(rows).set_index(["方法", "资产"]).style.format(
    {"方差误差": "{:+.1%}", "偏度（真实）": "{:.2f}", "偏度（模拟）": "{:.2f}",
     "峰度（真实）": "{:.2f}", "峰度（模拟）": "{:.2f}", "KS 距离": "{:.3f}"})

**判读**：递推式对第 1 位资产没有影响，但第 2、3 位的方差分别被高估约 33% 和 22%，
峰度被明显稀释（尾部变薄），KS 距离是正确方法的 4–5 倍。两种正确方法的方差误差都在几个百分点内
（来自 KDE 带宽本身的平滑），偏度、峰度基本保持。

---
## 2 · 真实数据：误差有多大？

每季度末取过去 120 个交易日，各方法模拟 2 万次，与该窗口的历史数据比较。

In [4]:
def distortion(R, order=None, step="Q"):
    cols = list(R.columns) if order is None else order
    ends = R.groupby(R.index.to_period(step)).tail(1).index[1:]
    rows = []
    for t in ends:
        H = R.loc[:t, cols].tail(WINDOW).values
        C0 = np.cov(H, rowvar=False)
        for name, f in sm.METHODS.items():
            X = f(H, 20_000, rng)
            C = np.cov(X, rowvar=False)
            for j, a in enumerate(cols):
                rows.append({"方法": name, "资产": a, "方差误差": C[j, j] / C0[j, j] - 1,
                             "KS 距离": ks_2samp(X[:, j], H[:, j]).statistic})
    return pd.DataFrame(rows)

d2 = distortion(R2)
d2.pivot_table(index="资产", columns="方法", values="方差误差", aggfunc="median").style.format("{:+.1%}")

方法,Iman-Conover,国泰君安递推,高斯Copula+KDE
资产,,,
中债综合财富,+4.7%,+7.3%,+5.0%
沪深300,+7.0%,+7.5%,+7.6%


**股债两资产**：股债日收益相关接近 0，递推式里注入相关的那一项几乎为零，三种方法的误差相同
（约 6–7% 的方差放大来自 KDE 带宽，三者共有）。**递推式的缺陷在股债配置上几乎没有影响。**

In [5]:
d5 = distortion(R5)
d5r = distortion(R5, order=ASSETS5[::-1])
tbl = pd.concat({
    "方差误差（原顺序）": d5.pivot_table(index="资产", columns="方法", values="方差误差", aggfunc="median"),
    "方差误差（倒序排列）": d5r.pivot_table(index="资产", columns="方法", values="方差误差", aggfunc="median"),
    "KS 距离（原顺序）": d5.pivot_table(index="资产", columns="方法", values="KS 距离", aggfunc="median"),
}, axis=1).reindex(ASSETS5)
tbl.style.format("{:+.1%}", subset=[c for c in tbl.columns if "方差" in c[0]]).format(
    "{:.3f}", subset=[c for c in tbl.columns if "KS" in c[0]])

**五资产**：恒生、标普 500 与沪深 300 相关较高，递推式把它们的方差分别高估约 52% 和 37%。
把资产顺序倒过来排，原本排第一、不受影响的沪深 300 变成排在最后，方差被高估约 70% ——
误差落在谁身上，取决于资产在列表里的位置。
两种正确方法的误差与资产顺序无关。

---
## 3 · 对配置的影响

在四个风险资产（沪深300、恒生、标普500、黄金）上，每半年末用各方法模拟 1500 条 20 日路径，
求期末收益 ES 最小的权重。「高斯 Copula 重抽」是同一方法换一组随机数重新模拟，
用来衡量蒙特卡洛噪声本身会让权重差多少。

In [6]:
RISKY = ["沪深300ETF", "恒生ETF", "标普500ETF", "黄金ETF"]
Rr = R5[RISKY]

def min_es_weights(method, H, order):
    P = sm.simulate_horizon(method, H[:, order], HORIZON, N_PATHS, rng).sum(axis=1)
    w = al.min_cvar(pd.DataFrame(P), q=0.95, upper=1.0).values
    out = np.empty(len(order)); out[order] = w
    return out

variants = {
    "高斯 Copula 重抽": (sm.gaussian_copula_kde, [0, 1, 2, 3]),
    "Iman-Conover": (sm.iman_conover, [0, 1, 2, 3]),
    "递推（原顺序：沪深300 在前）": (sm.gtja_recursion, [0, 1, 2, 3]),
    "递推（倒序：黄金在前）": (sm.gtja_recursion, [3, 2, 1, 0]),
}
rows = []
for t in Rr.groupby(Rr.index.to_period("2Q")).tail(1).index[1:]:
    H = Rr.loc[:t].tail(WINDOW).values
    ref = min_es_weights(sm.gaussian_copula_kde, H, [0, 1, 2, 3])
    for name, (f, order) in variants.items():
        w = min_es_weights(f, H, order)
        rows.append({"方法": name, "与高斯 Copula 的权重差（单边）": np.abs(w - ref).sum() / 2, **dict(zip(RISKY, w))})
partW = pd.DataFrame(rows).groupby("方法").mean().loc[list(variants)]
partW.style.format("{:.1%}")

,与高斯 Copula 的权重差（单边）,沪深300ETF,恒生ETF,标普500ETF,黄金ETF
方法,,,,,
高斯 Copula 重抽,6.2%,19.0%,8.3%,28.8%,43.9%
Iman-Conover,6.5%,19.9%,7.2%,29.3%,43.7%
递推（原顺序：沪深300 在前）,8.6%,23.2%,6.4%,24.8%,45.6%
递推（倒序：黄金在前）,9.7%,12.9%,8.4%,31.5%,47.2%


**判读**：同一方法换一组随机数，权重本身就会差约 6%（蒙特卡洛噪声）；递推式与高斯 Copula 的差异约 9–10%，
比噪声多出约一半。更重要的是方向是系统性的：它多配排在第一位的资产（第一位资产的风险没被高估，后面的被高估了），
同一组数据只因排列顺序不同，沪深 300 的平均权重就从约 23% 变成约 13%。

---
## 4 · 样本外回测

四个风险资产，每月末用过去 120 个交易日，按各方法求 20 日期末 ES 最小的权重，持有一个月，不计成本。
对照：不做模拟、直接用 120 日历史日收益求最小 ES；最小方差；等权。

In [7]:
def sim_strategy(method, order=None):
    def f(W):
        o = list(range(W.shape[1])) if order is None else order
        return pd.Series(min_es_weights(method, W.values, o), W.columns)
    return f

STRATS = {
    "递推（原顺序）": sim_strategy(sm.gtja_recursion),
    "递推（倒序）": sim_strategy(sm.gtja_recursion, [3, 2, 1, 0]),
    "高斯 Copula + KDE": sim_strategy(sm.gaussian_copula_kde),
    "Iman-Conover": sim_strategy(sm.iman_conover),
    "历史最小ES（不模拟）": lambda W: al.min_cvar(W, q=0.95, upper=1.0),
    "最小方差": lambda W: al.min_variance(W, upper=1.0),
    "等权": al.equal_weights,
}
port, weights = al.rolling_backtest(Rr, STRATS, lookback=WINDOW)
res = {}
for c in port:
    s = bt.perf_stats(port[c])
    s["样本外日 ES95"] = es_hist(port[c])
    s["月均单边换手"] = al.turnover(weights[c])
    res[c] = s
print(f"回测区间 {port.index.min():%Y-%m-%d} ~ {port.index.max():%Y-%m-%d}")
pd.DataFrame(res).T.style.format({"年化收益": "{:.2%}", "年化波动": "{:.2%}", "最大回撤": "{:.2%}",
                                   "样本外日 ES95": "{:.2%}", "月均单边换手": "{:.1%}",
                                   "收益波动比": "{:.2f}", "超额夏普": "{:.2f}", "卡玛比率": "{:.2f}"})

回测区间 2014-08-01 ~ 2026-10-12


,年化收益,年化波动,收益波动比,超额夏普,最大回撤,卡玛比率,样本外日 ES95,月均单边换手
递推（原顺序）,12.51%,10.89%,1.15,0.97,-16.00%,0.78,1.65%,12.6%
递推（倒序）,13.23%,10.85%,1.22,1.03,-17.62%,0.75,1.65%,12.7%
高斯 Copula + KDE,12.81%,10.85%,1.18,1.00,-16.23%,0.79,1.65%,13.6%
Iman-Conover,12.47%,10.89%,1.15,0.96,-16.31%,0.76,1.66%,13.1%
历史最小ES（不模拟）,11.46%,11.05%,1.04,0.86,-17.25%,0.66,1.61%,14.4%
最小方差,12.34%,10.11%,1.22,1.02,-16.02%,0.77,1.50%,8.3%
等权,9.25%,12.34%,0.75,0.59,-19.56%,0.47,1.88%,0.0%


In [8]:
pairs = [("递推（原顺序）", "高斯 Copula + KDE"), ("递推（倒序）", "高斯 Copula + KDE"),
         ("高斯 Copula + KDE", "历史最小ES（不模拟）")]
rows = {}
for a, b in pairs:
    pt, lo, hi, p = al.block_bootstrap_sharpe_diff(port[a], port[b])
    rows[f"{a} − {b}"] = {"超额夏普差": pt, "95% 下限": lo, "95% 上限": hi}
pd.DataFrame(rows).T.style.format("{:+.2f}")

,超额夏普差,95% 下限,95% 上限
递推（原顺序） − 高斯 Copula + KDE,-0.03,-0.11,+0.07
递推（倒序） − 高斯 Copula + KDE,+0.03,-0.06,+0.13
高斯 Copula + KDE − 历史最小ES（不模拟）,+0.12,-0.07,+0.31


**判读**：样本外各方法的差别很小，都不显著。递推式的缺陷虽然在模拟层面真实存在、并且改变了权重，
但在这组资产和这个风险度量下，没有转化为可检验的样本外差异。
用 KDE 模拟比直接用历史日收益求最小 ES 的超额夏普高约 0.12，但同样不显著；
简单的最小方差组合与各种模拟方法表现相当。

---
# 结论

| 问题 | 结论 |
|---|---|
| 递推式能否保持协方差和边缘分布 | **不能**。已知答案的数据上，第 2、3 位资产方差高估 33%、22%，峰度被稀释，KS 距离是正确方法的 4–5 倍 |
| 真实数据上影响多大 | **取决于相关性**。股债（相关≈0）几乎无影响；五资产中恒生、标普 500 方差高估约 52%、37% |
| 有没有额外的副作用 | **结果依赖资产排列顺序**。误差落在排位靠后、且与前面资产相关的资产上；把沪深 300 排到最后，它的方差被高估约 70% |
| 是否改变配置 | **会，但幅度有限**。权重差异约 9–10%，比蒙特卡洛噪声（约 6%）多出约一半；方向上系统性多配排在第一位的资产 |
| 是否影响样本外表现 | 在本研究的设定下**不显著** |
| 正确做法 | 高斯 Copula + KDE 分位数逆变换，或 Iman-Conover 重排；两者都保持边缘分布、与顺序无关 |

原文的收益数字（股债年化 7.06%、五资产年化 10.1%）依赖未定义的「期内 + 期末风险」度量，本研究无法复现。